### **Introduction and Objectives**

This notebook loads the core AuroraPay datasets (transactions, risks, controls, incidents, issues), performs light data quality checks and standardization, and produces cleaned versions for downstream analysis and dashboarding. It also computes a first set of descriptive metrics (volumes, fraud rates, incident and issue counts) to validate that the synthetic data behaves in a realistic way.

In [4]:
import pandas as pd
import numpy as np

pd.options.display.float_format = "{:,.4f}".format

**Step 1 – Load raw datasets**

Load the CSV files from data/raw/ into pandas DataFrames.

In [6]:
transactions_raw = pd.read_csv('../data/raw/transactions.csv', parse_dates=['timestamp'])
risk_register = pd.read_csv('../data/raw/risk_register.csv')
controls = pd.read_csv('../data/raw/controls.csv')
incidents = pd.read_csv('../data/raw/incidents.csv', parse_dates=['start_time', 'end_time'])
issues = pd.read_csv('../data/raw/issues.csv', parse_dates=['opened_date', 'target_resolution_date'])

transactions_raw.head()

,transaction_id,customer_id,channel,product,amount,currency,timestamp,status,fraud_flag
0,1,15796,POS,Debit,90.6900,CAD,2026-03-04,Success,0
1,2,861,Online,e-Transfer,35.2900,CAD,2026-03-10,Success,0
2,3,5391,POS,Debit,15.5000,CAD,2026-03-25,Success,0
3,4,11965,Online,e-Transfer,11.5200,CAD,2026-03-12,Success,0
4,5,11285,Online,Debit,321.8800,CAD,2026-01-25,Success,0


In [9]:
transactions_raw.shape, risk_register.shape, controls.shape, issues.shape, incidents.shape

((100000, 9), (12, 14), (8, 12), (8, 9), (8, 11))

**Step 2 – Basic data quality checks**

Perform high‑level sanity checks on each dataset (row counts, missing values, category distributions).

In [12]:
# Transactions Overview
print(transactions_raw.describe())

#Check Categories
print(transactions_raw['product'].value_counts())
print(transactions_raw['channel'].value_counts())
print(transactions_raw['status'].value_counts())
print(transactions_raw['fraud_flag'].value_counts())

       transaction_id  customer_id       amount                   timestamp  \
count    100,000.0000 100,000.0000 100,000.0000                      100000   
mean      50,000.5000  10,027.9796      27.5930  2026-02-14 01:18:25.343999   
min            1.0000       1.0000       1.0000         2026-01-01 00:00:00   
25%       25,000.7500   5,021.0000      11.6900         2026-01-23 00:00:00   
50%       50,000.5000  10,042.0000      19.9800         2026-02-14 00:00:00   
75%       75,000.2500  15,043.2500      34.3000         2026-03-08 00:00:00   
max      100,000.0000  20,000.0000     587.2700         2026-03-30 00:00:00   
std       28,867.6578   5,774.5523      25.9084                         NaN   

        fraud_flag  
count 100,000.0000  
mean        0.0015  
min         0.0000  
25%         0.0000  
50%         0.0000  
75%         0.0000  
max         1.0000  
std         0.0383  
product
Debit         59798
e-Transfer    40202
Name: count, dtype: int64
channel
POS       39952
O

Check for missing values:

In [16]:
print('Transactions: \n', transactions_raw.isna().sum())
print('Risk Register: \n', risk_register.isna().sum())
print('Controls: \n', controls.isna().sum())
print('Incidents: \n', incidents.isna().sum())
print('Issues: \n', issues.isna().sum())

Transactions: 
 transaction_id    0
customer_id       0
channel           0
product           0
amount            0
currency          0
timestamp         0
status            0
fraud_flag        0
dtype: int64
Risk Register: 
 risk_id                    0
risk_category              0
risk_subcategory           0
risk_name                  0
description                0
inherent_likelihood        0
inherent_impact            0
inherent_score             0
residual_likelihood        0
residual_impact            0
residual_score             0
risk_appetite_statement    0
risk_owner                 0
status                     0
dtype: int64
Controls: 
 control_id                 0
risk_id                    0
control_name               0
control_description        0
control_type               0
control_owner              0
frequency                  0
design_effectiveness       0
operating_effectiveness    0
key_control                0
last_test_date             0
comments                

Quick checks on incidents and issues:

In [17]:
print(incidents["severity"].value_counts())
print(issues["severity"].value_counts())
print(issues["status"].value_counts())

severity
High        5
Medium      2
Critical    1
Name: count, dtype: int64
severity
High        5
Medium      2
Critical    1
Name: count, dtype: int64
status
Open           6
In Progress    2
Name: count, dtype: int64


**Step 3 – Light cleaning and standardization**

Ensure column types and categorical values are consistent across datasets. For this case study, we expect minimal cleaning, but we still enforce some basic standards.

In [18]:
# Ensure fraud_flag is integer 0/1
transactions_clean = transactions_raw.copy()
transactions_clean["fraud_flag"] = transactions_clean["fraud_flag"].fillna(0).astype(int)

# Standardize product and status strings (strip whitespace, title case if needed)
transactions_clean["product"] = transactions_clean["product"].str.strip()
transactions_clean["channel"] = transactions_clean["channel"].str.strip()
transactions_clean["status"] = transactions_clean["status"].str.strip()

# Derive simple date fields for later analysis
transactions_clean["date"] = transactions_clean["timestamp"].dt.date
transactions_clean["month"] = transactions_clean["timestamp"].dt.to_period("M").astype(str)

For incidents:

In [19]:
incidents_clean = incidents.copy()
incidents_clean["duration_minutes"] = (
    (incidents_clean["end_time"] - incidents_clean["start_time"])
    .dt.total_seconds() / 60
)

# Basic check: duration >= 0
(incidents_clean["duration_minutes"] < 0).sum()

np.int64(0)

**Step 4 – Save processed datasets**

Save cleaned versions to data/processed/ for use in later notebooks and Power BI.

In [22]:
transactions_clean.to_csv("../data/processed/transactions_clean.csv", index=False)
incidents_clean.to_csv("../data/processed/incidents_clean.csv", index=False)

# Risk register, controls, and issues may be unchanged, but we can still create 'processed' copies for consistency.
risk_register.to_csv("../data/processed/risk_register.csv", index=False)
controls.to_csv("../data/processed/controls.csv", index=False)
issues.to_csv("../data/processed/issues.csv", index=False)

**Step 5 – Initial descriptive metrics**

Compute a small set of descriptive metrics to validate the data and to use as inputs to dashboards and the executive risk narrative.

Transactions volume and counts

In [27]:
# Overall
total_txn = len(transactions_clean)
txn_total_volume = transactions_clean["amount"].sum()
print('Total Transactions: ', total_txn)
print("Total Transactions Volume: ", txn_total_volume)

# Transactions by product
txn_by_product = transactions_clean.groupby("product").agg(
    txn_count = ('transaction_id', 'count'),
    txn_volume = ('amount', 'sum'),
    fraud_txn = ('fraud_flag', 'sum')
)

txn_by_product['fraud_rate'] = txn_by_product['fraud_txn'] / txn_by_product['txn_count']
txn_by_product

Total Transactions:  100000
Total Transactions Volume:  2759295.96


,txn_count,txn_volume,fraud_txn,fraud_rate
product,,,,
Debit,59798,"1,653,687.2000",31,0.0005
e-Transfer,40202,"1,105,608.7600",116,0.0029


In [28]:
# Transactions by channel
txn_by_channel = (
    transactions_clean.groupby('channel').agg(
        txn_count = ('transaction_id', 'count'),
        txn_volume = ('amount', 'sum')
    )
)
txn_by_channel

,txn_count,txn_volume
channel,,
Mobile,24971,"689,030.4500"
Online,35077,"966,889.1900"
POS,39952,"1,103,376.3200"


Incidents by risk and severity

In [29]:
incidents_by_risk = (
    incidents_clean.groupby(['risk_id', 'severity']).agg(
        incident_count = ('incident_id', 'count'),
        total_duration_minutes = ('duration_minutes', 'sum')
    ).reset_index()
)

incidents_by_risk

,risk_id,severity,incident_count,total_duration_minutes
0,R1,Critical,1,57.0000
1,R11,Medium,1,"15,839.9833"
2,R2,High,1,95.0000
3,R3,High,1,"24,479.9833"
4,R4,Medium,1,"12,959.9833"
5,R5,High,1,105.0000
6,R7,High,1,45.0000
7,R9,High,1,"20,095.0000"


In [31]:
incidents_with_category = incidents_by_risk.merge(
    risk_register[['risk_id', 'risk_category']],
    on='risk_id',
    how='left'
)

incidents_with_category

,risk_id,severity,incident_count,total_duration_minutes,risk_category
0,R1,Critical,1,57.0000,Operational
1,R11,Medium,1,"15,839.9833",Technology & Security
2,R2,High,1,95.0000,Operational
3,R3,High,1,"24,479.9833",Fraud
4,R4,Medium,1,"12,959.9833",Fraud
5,R5,High,1,105.0000,Technology & Security
6,R7,High,1,45.0000,Third-Party
7,R9,High,1,"20,095.0000",Regulatory & Oversight


Issues by risk and severity

In [32]:
issues_by_risk = (
    issues.groupby(['risk_id', 'severity', 'status']).agg(
        issues_count = ('issue_id', 'count')
    )
    .reset_index()
)

issues_by_risk

,risk_id,severity,status,issues_count
0,R11,High,Open,1
1,R12,High,Open,1
2,R3,High,In Progress,1
3,R3,Medium,Open,1
4,R5,Critical,In Progress,1
5,R6,Medium,Open,1
6,R7,High,Open,1
7,R9,High,Open,1


Save these summary tables for later analysis:

In [34]:
txn_by_product.to_csv("../data/processed/txn_by_product.csv")
txn_by_channel.to_csv("../data/processed/txn_by_channel.csv")
incidents_with_category.to_csv("../data/processed/incidents_by_risk.csv", index=False)
issues_by_risk.to_csv("../data/processed/issues_by_risk.csv", index=False)

**Notebook Summary: **

Fraud rates are higher on e‑Transfer than Debit, as expected. Incident volumes are concentrated in operational and fraud risks, with a handful of High/Critical events. Several High/Critical issues remain open for privileged access and vendor resilience, which will be important themes in the executive narrative.